In [1]:
import json
import pandas as pd
import h5py
import numpy as np
import tensorflow as tf
import keras
from tensorflow.keras.layers import Dense, GRU
from tensorflow.keras.models import load_model

model = load_model('./Quickdraw5Class1_20.h5')

2026-01-25 18:11:41.661142: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-25 18:11:41.706629: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-25 18:11:41.706668: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-25 18:11:41.707750: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-25 18:11:41.714478: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-25 18:11:41.714977: I tensorflow/core/platform/cpu_feature_guard.cc:1

In [2]:
for layer in model.layers:
    if layer.weights: # Check if the layer has trainable weights (like Dense or Conv2D)
        print(f"{layer.name}:")
        if isinstance(layer, GRU):
            input_weights, hidden_weights, biases = layer.get_weights()
            print(f"input weights shape: {input_weights.shape}")
            print(f"hidden weights shape: {hidden_weights.shape}")
            print(f"biases shape: {biases.shape}")
        elif isinstance(layer, Dense):
            weights, biases = layer.get_weights()
            print(f"weights shape: {weights.shape}")
            print(f"biases shape: {biases.shape}")
            # print(f"Weights: {weights}") # Uncomment to see the actual values
            print(f"Biases: {biases}") # actual values

gru:
input weights shape: (3, 96)
hidden weights shape: (32, 96)
biases shape: (2, 96)
dense:
weights shape: (3200, 32)
biases shape: (32,)
Biases: [ 0.09225596 -0.22272529 -0.04148769 -0.15092641 -0.28090882  0.01596832
 -0.06267665  0.00168173 -0.1805329   0.09314916 -0.01221799 -0.10496804
  0.1431821  -0.03500383  0.11908529  0.160275   -0.01848121 -0.11066682
 -0.4120701  -0.03423632 -0.06231544 -0.33608502  0.08323162 -0.16292156
 -0.29581064 -0.21686606 -0.02555476 -0.20755582 -0.00757259 -0.18589126
  0.01918316 -0.02120592]
dense_1:
weights shape: (32, 16)
biases shape: (16,)
Biases: [1.0911219  0.69869304 0.9770719  1.5308261  1.6354897  0.75624347
 1.59204    1.3113813  1.0595832  1.5675517  1.7303895  1.8470563
 1.0737954  2.0047438  1.7986232  0.79785705]
rnn_densef:
weights shape: (16, 5)
biases shape: (5,)
Biases: [-0.1332287   0.05734754 -0.7944131   0.8292614  -0.37966573]


In [3]:
# Extract weights and biases for each layer (gru layer separated to 3 dense)
# and print the weights to output and to files with name layer_weight.txt and layer_bias.txt
for layer in model.layers:
    print(layer.name)
    if isinstance(layer, GRU):

        print(layer.get_config())

        weights = layer.get_weights()
        print("Number of weights tensors: " + str(len(weights)))

        # W for x, U for h, B for biases
        W = weights[0]
        U = weights[1]
        B = weights[2]

        print("input size " + str(W.shape[0]))
        print("hidden state size " + str(layer.units))

        # extract three weights and biases for the three dense layers

        reset_gate_weight = np.concatenate((W[:, :layer.units], U[:, :layer.units]), axis = 0)
        update_gate_weight = np.concatenate((W[:, layer.units:2*layer.units], U[:, layer.units:2*layer.units]), axis=0)
        candidate_gate_weight = np.concatenate((W[:, 2*layer.units:], U[:, 2*layer.units:]), axis=0)

        # combine input biases and recurrent biases
        # notice recurrent biases is used when reset_after config set to True for the gru layer (default is True)
        reset_gate_bias = np.sum(B[:, :layer.units], axis = 0)
        update_gate_bias = np.sum(B[:, layer.units:2*layer.units], axis = 0)
        candidate_gate_bias = np.sum(B[:, 2*layer.units:], axis = 0)

        print("reset_gate_weight")
        print(reset_gate_weight.shape)
        print("update_gate_weight")
        print(update_gate_weight.shape)
        print("candidate_gate_weight")
        print(candidate_gate_weight.shape)

        print("reset_gate_bias")
        print(reset_gate_bias.shape)
        print("update_gate_bias")
        print(update_gate_bias.shape)
        print("candidate_gate_bias")
        print(candidate_gate_bias.shape)


        # find maximum and minimum values for each weight and bias
        print("max and min values")
        print("reset_gate_weight")
        print(np.max(reset_gate_weight))
        print(np.min(reset_gate_weight))
        print("update_gate_weight")
        print(np.max(update_gate_weight))
        print(np.min(update_gate_weight))
        print("candidate_gate_weight")
        print(np.max(candidate_gate_weight))
        print(np.min(candidate_gate_weight))

        print("reset_gate_bias")
        print(np.max(reset_gate_bias))
        print(np.min(reset_gate_bias))
        print("update_gate_bias")
        print(np.max(update_gate_bias))
        print(np.min(update_gate_bias))
        print("candidate_gate_bias")
        print(np.max(candidate_gate_bias))
        print(np.min(candidate_gate_bias))
        

        # write to npy files
        np.savetxt('reset_gate_weights.txt', reset_gate_weight.flatten())
        np.savetxt('update_gate_weights.txt', update_gate_weight.flatten())
        np.savetxt('candidate_gate_weights.txt', candidate_gate_weight.flatten())

        np.savetxt('reset_gate_biases.txt', reset_gate_bias.flatten())
        np.savetxt('update_gate_biases.txt', update_gate_bias.flatten())
        np.savetxt('candidate_gate_biases.txt', candidate_gate_bias.flatten())


    elif isinstance(layer, Dense):
        # all other layers are dense with relu activation
        
        # print layer info (activation)
        print(layer.get_config())
        weights, biases = layer.get_weights()

        # print weights and biases
        print("weights shape")
        print(weights.shape)
        print("biases shape")
        print(biases.shape)

        # find maximum and minimum values for each weight and bias
        print("max and min values")
        print("weights")
        print(np.max(weights))
        print(np.min(weights))
        print("biases")
        print(np.max(biases))
        print(np.min(biases))    

        # # write to a txt file, flatten the weights
        np.savetxt(layer.name + '_weights.txt', weights.flatten())
        np.savetxt(layer.name + '_biases.txt', biases.flatten())
    
    print("=====================================================================================================")

input_1
gru
{'name': 'gru', 'trainable': True, 'dtype': 'float32', 'return_sequences': True, 'return_state': False, 'go_backwards': False, 'stateful': False, 'unroll': False, 'time_major': False, 'units': 32, 'activation': 'tanh', 'recurrent_activation': 'sigmoid', 'use_bias': True, 'kernel_initializer': {'module': 'keras.initializers', 'class_name': 'GlorotUniform', 'config': {'seed': None}, 'registered_name': None}, 'recurrent_initializer': {'module': 'keras.initializers', 'class_name': 'Orthogonal', 'config': {'gain': 1.0, 'seed': None}, 'registered_name': None}, 'bias_initializer': {'module': 'keras.initializers', 'class_name': 'Zeros', 'config': {}, 'registered_name': None}, 'kernel_regularizer': None, 'recurrent_regularizer': None, 'bias_regularizer': None, 'activity_regularizer': None, 'kernel_constraint': None, 'recurrent_constraint': None, 'bias_constraint': None, 'dropout': 0.0, 'recurrent_dropout': 0.0, 'implementation': 2, 'reset_after': True}
Number of weights tensors: 3
i